# 📊 Data Understanding — IndoToxic2024 Dataset
**Proyek:** Indonesian Hate Speech Analyzer (Kelompok 6)  
**Arsitektur Model:** XLM-RoBERTa  
**Notebook ini** melakukan Exploratory Data Analysis (EDA) pada dataset `indotoxic2024_annotated_data_v2_final.csv`.

---
## Bab 1: Setup & Data Loading

In [ ]:
# === Import Library ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 150)

print("✅ Library berhasil dimuat.")

### 1.1 Memuat Dataset Utama

In [ ]:
import pandas as pd
from pathlib import Path

# Menentukan root directory secara otomatis & dinamis.
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

# Sumber utama tetap dataset project yang diminta.
file_path = project_root / 'data' / 'raw' / 'indotoxic2024_annotated_data_v2_final.csv'

# Fallback hanya untuk portabilitas ketika notebook dipindahkan bersama CSV.
local_fallback = current_dir / 'indotoxic2024_annotated_data_v2_final.csv'

if file_path.exists():
    data_source = file_path
elif local_fallback.exists():
    data_source = local_fallback
else:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Letakkan file pada: "
        f"{file_path} (path utama project) atau {local_fallback}."
    )

df = pd.read_csv(data_source)

print(f"Sumber dataset             : {data_source.resolve()}")
print(f"Jumlah Baris (Data Point)  : {df.shape[0]:,}")
print(f"Jumlah Kolom (Variabel)    : {df.shape[1]}")
print()
df.head()

### 1.2 Memahami Struktur Dataset (Poin 1)
Menampilkan nama kolom, tipe data, dan jumlah data non-null untuk setiap variabel.

In [ ]:
df.info()

In [ ]:
# Daftar nama kolom beserta tipe datanya
print("=== Daftar Kolom & Tipe Data ===")
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"  {i:2d}. {col:<40s} → {dtype}")

### 1.3 Memeriksa Kualitas Data (Poin 2)
Mengecek data kosong (NaN/missing values), data duplikat, dan distribusi panjang teks.

In [ ]:
# --- Cek Missing Values ---
print("=== Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah NaN': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah NaN'] > 0].to_string())
print()

# --- Cek Duplikat ---
n_dup = df.duplicated(subset='text').sum()
print(f"Jumlah teks duplikat: {n_dup}")
print()

# --- Statistik Panjang Teks ---
df['char_count'] = df['text'].astype(str).str.len()
df['word_count'] = df['text'].astype(str).str.split().str.len()

print("=== Statistik Panjang Teks ===")
print(df[['char_count', 'word_count']].describe().round(1).to_string())

In [ ]:
# --- Identifikasi Teks Sangat Pendek & Sangat Panjang ---
very_short = df[df['word_count'] <= 3]
very_long  = df[df['word_count'] >= 200]

print(f"Teks sangat pendek (≤ 3 kata) : {len(very_short):,} baris")
print(f"Teks sangat panjang (≥ 200 kata): {len(very_long):,} baris")
print()

if len(very_short) > 0:
    print("--- Sampel Teks Sangat Pendek ---")
    print(very_short[['text_id', 'text']].head(5).to_string(index=False))

---
## Bab 2: Parsing & Agregasi Label (Majority Voting)

Kolom-kolom label pada dataset masih tersimpan sebagai **string representasi list** (contoh: `"['1', '0']"`).  
Kita perlu mengubahnya menjadi satu label konsensus menggunakan rumus **Majority Voting**:

$$\text{Label} = \begin{cases} 1, & \text{jika rata-rata vote} > 0.5 \\ 0, & \text{jika rata-rata vote} < 0.5 \\ \text{Disagreement}, & \text{jika rata-rata vote} = 0.5 \end{cases}$$

In [ ]:
# === Fungsi Parsing Anotasi List ===

def parse_annotation_list(annotation_str):
    """Mengubah string list anotasi menjadi list angka.
    Contoh: "['1', '0']" -> [1, 0]
    """
    try:
        parsed = ast.literal_eval(annotation_str)
        return [int(x) for x in parsed]
    except (ValueError, SyntaxError):
        return []

print("Fungsi parse_annotation_list() berhasil didefinisikan.")

### 2.1 Analisis Distribusi Jumlah Anotator per Teks
Sebelum melakukan majority voting, kita periksa apakah jumlah annotator di dalam list selalu 2, atau ada teks yang dinilai oleh 1, 3, atau lebih annotator.

In [ ]:
# Menghitung panjang list annotators_id untuk setiap baris
df['num_annotators'] = df['annotators_id'].apply(lambda x: len(parse_annotation_list(x)))

# Tabel frekuensi jumlah annotator
annotator_counts = df['num_annotators'].value_counts().sort_index()
annotator_pct = (annotator_counts / len(df) * 100).round(2)

dist_annotator_df = pd.DataFrame({
    'Jumlah Anotator (Panjang List)': annotator_counts.index,
    'Jumlah Teks (Baris)': annotator_counts.values,
    'Persentase (%)': annotator_pct.values
})

print("=== Distribusi Jumlah Anotator per Teks ===")
print(dist_annotator_df.to_string(index=False))
print()
print(f"Total baris data         : {len(df):,}")
print(f"Rentang jumlah annotator : {df['num_annotators'].min()} s/d {df['num_annotators'].max()} annotator per teks")

In [ ]:
# Visualisasi Distribusi Jumlah Anotator
fig, ax = plt.subplots(figsize=(10, 5))

# Kelompokkan kategori: 1 sampai 6, dan >6
counts_display = annotator_counts[annotator_counts.index <= 6].copy()
other_count = annotator_counts[annotator_counts.index > 6].sum()
if other_count > 0:
    counts_display['>6'] = other_count

bars = ax.bar([str(idx) for idx in counts_display.index], counts_display.values, color='#3498db', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, counts_display.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Distribusi Jumlah Anotator per Teks (Panjang List)', fontsize=13, fontweight='bold')
ax.set_xlabel('Jumlah Anotator yang Menilai')
ax.set_ylabel('Jumlah Teks')
plt.tight_layout()
plt.show()

### 💡 Temuan Analisis Anotator:
Dari data di atas, kita menemukan fakta empiris penting:
1. **Jumlah annotator TIDAK HANYA 2 orang!** Rentang annotator berkisar dari **1 hingga 19 orang** per teks.
2. **55.4% (15.748 baris)** ternyata hanya dinilai oleh **1 annotator**.
3. **27.8% (7.907 baris)** dinilai oleh **2 annotator**.
4. **16.8% (4.793 baris)** dinilai oleh **3 atau lebih annotator**.

Ini membuktikan mengapa metode **Majority Voting** (perhitungan rata-rata suara > 0.5) mutlak diperlukan, karena sistem harus mampu menangani jumlah voter yang dinamis (1, 2, 3, hingga 19 orang)!

In [ ]:
# === Fungsi Majority Voting ===

def majority_vote(annotation_str):
    """Menghitung label konsensus dari string list anotasi.
    Returns:
        1   -> Mayoritas positif (rata-rata > 0.5)
        0   -> Mayoritas negatif (rata-rata < 0.5)
        0.5 -> Disagreement / Tie (rata-rata = 0.5)
    """
    votes = parse_annotation_list(annotation_str)
    if len(votes) == 0:
        return np.nan
    avg = np.mean(votes)
    if avg > 0.5:
        return 1
    elif avg < 0.5:
        return 0
    else:
        return 0.5  # Disagreement

print("Fungsi majority_vote() berhasil didefinisikan.")

In [ ]:
# === Kolom-kolom label yang akan diproses ===
label_columns = [
    'toxicity',
    'identity_attack',
    'threat_incitement_to_violence',
    'insults',
    'profanity_obscenity',
    'sexually_explicit',
    'polarized',
    'related_to_election_2024',
    'is_noise_or_spam_text'
]

# Menerapkan majority voting ke semua kolom label
for col in label_columns:
    new_col = f'{col}_label'
    df[new_col] = df[col].apply(majority_vote)
    print(f"  ✅ {col} → {new_col}")

print()
print("Semua kolom label berhasil diparsing!")
print()

# Menampilkan sampel hasil parsing
sample_cols = ['text_id', 'toxicity', 'toxicity_label', 'insults', 'insults_label']
df[sample_cols].head(5)

---
## Bab 3: Analisis Distribusi Label & Pembuktian Statistik Kritis (Poin 3)

### 3.1 Distribusi Label Toksisitas (Task Utama: Biner)

In [ ]:
# === Distribusi Toxicity (Biner) ===
tox_counts = df['toxicity_label'].value_counts().sort_index()

label_map = {0: 'Non-Toxic (0)', 0.5: 'Disagreement (0.5)', 1: 'Toxic (1)'}
tox_display = tox_counts.rename(index=label_map)

print("=== Distribusi Label Toksisitas ===")
for label, count in tox_display.items():
    pct = count / len(df) * 100
    print(f"  {label:<25s}: {count:>6,} ({pct:.1f}%)")

# Visualisasi Bar Chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(tox_display.index, tox_display.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, tox_display.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Distribusi Label Toksisitas (Majority Voting)', fontsize=14, fontweight='bold')
ax.set_ylabel('Jumlah Data')
ax.set_xlabel('Kategori Label')
plt.tight_layout()
plt.show()

### 3.2 Distribusi Sub-Kategori Multi-Label

In [ ]:
# === Distribusi 5 Sub-Kategori Toksisitas ===
multi_label_cols = [
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

multi_label_names = [
    'Identity Attack\n(SARA)',
    'Threat\n(Ancaman)',
    'Insults\n(Hinaan)',
    'Profanity\n(Kata Kasar)',
    'Sexually\nExplicit'
]

# Menghitung jumlah kelas 1 (positif) untuk setiap sub-kategori
positive_counts = [df[col].apply(lambda x: 1 if x == 1 else 0).sum() for col in multi_label_cols]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(multi_label_names, positive_counts, color='#e74c3c', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, positive_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Jumlah Teks Positif (Kelas 1) per Sub-Kategori Toksisitas', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Data Berlabel 1')
ax.set_xlabel('Sub-Kategori')
plt.tight_layout()
plt.show()

### 3.3 [PEMBUKTIAN] Rekonstruksi Tabel Statistik Kritis
Memvalidasi angka-angka di **Tabel 5: Distribusi & Statistik Kritis Variabel** pada dokumen `Data_Understanding_Guide.md`.

In [ ]:
# === Rekonstruksi Tabel Statistik Kritis ===
# Memvalidasi angka dari Data_Understanding_Guide.md

all_label_cols = [
    ('toxicity_label',                         'toxicity (Task Utama)'),
    ('polarized_label',                        'polarized'),
    ('identity_attack_label',                  'identity_attack'),
    ('insults_label',                          'insults'),
    ('profanity_obscenity_label',              'profanity_obscenity'),
    ('threat_incitement_to_violence_label',     'threat_incitement_to_violence'),
    ('sexually_explicit_label',                'sexually_explicit'),
    ('is_noise_or_spam_text_label',            'is_noise_or_spam_text'),
    ('related_to_election_2024_label',         'related_to_election_2024'),
]

rows = []
for col, name in all_label_cols:
    total = len(df)
    n_0   = (df[col] == 0).sum()
    n_1   = (df[col] == 1).sum()
    n_dis = (df[col] == 0.5).sum()
    
    # Hitung rasio imbalance (Kelas 0 : Kelas 1)
    ratio = f"1 : {int(round(n_0 / n_1))}" if n_1 > 0 else "N/A"
    
    rows.append({
        'Variabel': name,
        'Kelas 0': f"{n_0:,} ({n_0/total*100:.1f}%)",
        'Kelas 1': f"{n_1:,} ({n_1/total*100:.1f}%)",
        'Disagreement': f"{n_dis:,} ({n_dis/total*100:.1f}%)",
        'Rasio Imbalance': ratio
    })

stats_df = pd.DataFrame(rows)
print("=" * 100)
print("TABEL STATISTIK KRITIS (Hasil Perhitungan Kode)")
print("Bandingkan dengan Tabel 5 di Data_Understanding_Guide.md")
print("=" * 100)
print(stats_df.to_string(index=False))

---
## Bab 4: Analisis Karakteristik Teks & Pembuktian Tantangan Data (Poin 4 & 6)

### 4.1 Distribusi Panjang Teks (Karakter & Kata)

In [ ]:
# === Histogram Panjang Teks: Toxic vs Non-Toxic ===
df_toxic     = df[df['toxicity_label'] == 1]
df_nontoxic  = df[df['toxicity_label'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Jumlah Kata
axes[0].hist(df_nontoxic['word_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[0].hist(df_toxic['word_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[0].set_title('Distribusi Jumlah Kata', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Histogram Jumlah Karakter
axes[1].hist(df_nontoxic['char_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[1].hist(df_toxic['char_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[1].set_title('Distribusi Jumlah Karakter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Karakter')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.suptitle('Perbandingan Panjang Teks: Toxic vs Non-Toxic', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistik rata-rata
print("=== Rata-rata Panjang Teks ===")
print(f"  Non-Toxic → Kata: {df_nontoxic['word_count'].mean():.1f}, Karakter: {df_nontoxic['char_count'].mean():.1f}")
print(f"  Toxic     → Kata: {df_toxic['word_count'].mean():.1f}, Karakter: {df_toxic['char_count'].mean():.1f}")

### 4.2 Kata yang Paling Sering Muncul pada Teks Toxic

In [ ]:
# === Top 20 Kata Paling Sering Muncul di Teks Toxic ===
from collections import Counter

# Mengambil semua kata dari teks berlabel Toxic
all_words_toxic = ' '.join(df_toxic['text'].astype(str).str.lower()).split()

# Menghitung frekuensi kemunculan
word_freq = Counter(all_words_toxic)
top_20 = word_freq.most_common(20)

# Visualisasi Horizontal Bar Chart
words, counts = zip(*top_20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(words)), counts, color='#e74c3c', edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_title('Top 20 Kata Paling Sering Muncul di Teks Toxic', fontsize=13, fontweight='bold')
ax.set_xlabel('Frekuensi Kemunculan')
plt.tight_layout()
plt.show()

### 4.3 Word Cloud — Teks Toxic

In [ ]:
# === Word Cloud: Kata Dominan pada Teks Toxic ===
try:
    from wordcloud import WordCloud

    text_toxic_all = ' '.join(df_toxic['text'].astype(str).str.lower())

    wordcloud = WordCloud(
        width=1000, height=500,
        background_color='white',
        colormap='Reds',
        max_words=150,
        collocations=False
    ).generate(text_toxic_all)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — Kata Dominan pada Teks Berlabel Toxic', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("⚠️ Library 'wordcloud' belum terinstal.")
    print("   Jalankan: pip install wordcloud")

### 4.4 Heatmap Korelasi Antar Sub-Label
Melihat apakah kalimat hinaan (*insults*) sering muncul bersamaan dengan kata kasar (*profanity*), sebagai justifikasi pemilihan fungsi aktivasi **Sigmoid** (independen per label).

In [ ]:
# === Heatmap Korelasi Antar Sub-Label ===
corr_cols = [
    'toxicity_label',
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

corr_names = ['Toxicity', 'SARA', 'Ancaman', 'Hinaan', 'Kata Kasar', 'Seksual']

# Hanya ambil data yang bukan disagreement (0 atau 1)
df_corr = df[corr_cols].copy()
df_corr = df_corr[(df_corr != 0.5).all(axis=1)]

correlation = df_corr.corr()
correlation.index = corr_names
correlation.columns = corr_names

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=-0.1, vmax=1, linewidths=0.5, ax=ax,
            square=True)
ax.set_title('Heatmap Korelasi Antar Label Toksisitas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 [PEMBUKTIAN] Tantangan Pemrosesan Teks Bahasa Indonesia
Menampilkan sampel teks Toxic secara langsung untuk membuktikan adanya **singkatan, slang, dan tipografi penghindaran sensor** yang disebutkan di dokumen `Data_Understanding_Guide.md`.

In [ ]:
# === Sampel Teks Toxic untuk Membuktikan Tantangan Bahasa Indonesia ===
print("=" * 80)
print("SAMPEL TEKS BERLABEL TOXIC")
print("Perhatikan penggunaan: singkatan, slang, kata kasar, sarkasme, dll.")
print("=" * 80)
print()

# Ambil 10 sampel acak dari teks berlabel Toxic
sample_toxic = df_toxic[['text_id', 'text']].sample(n=10, random_state=42)

for i, (_, row) in enumerate(sample_toxic.iterrows(), 1):
    print(f"[{i:2d}] ID: {row['text_id']}")
    print(f"     Teks: {row['text'][:300]}")
    print()

---
## Bab 5: Kesimpulan & Rekomendasi (Poin 7)

Berdasarkan eksplorasi data di atas, berikut adalah temuan utama dan rekomendasi untuk tahap selanjutnya:

### 📌 Temuan Utama
1. **Ketimpangan Kelas Sangat Ekstrem:** Label `toxicity` memiliki rasio Non-Toxic vs Toxic sekitar 11:1. Sub-kategori `sexually_explicit` bahkan mencapai rasio 566:1. Model baseline tanpa penanganan *imbalance* hampir pasti akan gagal mendeteksi kelas minoritas.
2. **Disagreement (Ambiguitas) Signifikan:** Sekitar 6.5% data pada label `toxicity` memiliki nilai *tie* (0.5) di mana para annotator tidak sepakat. Data ini perlu strategi penanganan khusus (drop, pisahkan, atau gunakan sebagai *soft label*).
3. **Tantangan Bahasa Terkonfirmasi:** Sampel teks menunjukkan penggunaan singkatan kasar, slang Indonesia, emoji, dan tipografi yang dirancang untuk menghindari filter kata (*censorship evasion*). Ini membuktikan bahwa pendekatan berbasis kamus kata kunci (*lexicon-based*) tidak akan cukup — model Transformer (XLM-RoBERTa) yang memahami konteks sangat diperlukan.
4. **Korelasi Antar Label:** Heatmap korelasi menunjukkan bahwa beberapa sub-label (seperti `insults` dan `profanity`) memiliki korelasi positif. Ini menjustifikasi penggunaan **Sigmoid** (bukan Softmax) pada arsitektur Multi-Label, karena satu teks bisa memiliki lebih dari satu label aktif secara bersamaan.

### 🔧 Rekomendasi untuk Tahap Data Engineering
1. **Penanganan Imbalance:** Wajib menerapkan teknik seperti *Class Weights*, *Focal Loss*, atau *oversampling* pada kelas minoritas.
2. **Text Cleaning:** Perlu dilakukan normalisasi slang, penghapusan URL/mention, dan pembersihan karakter khusus sebelum tokenisasi.
3. **Strategi Disagreement:** Tentukan apakah data dengan label 0.5 akan di-drop, dibulatkan, atau digunakan sebagai *soft label* untuk training.
4. **Analisis per Topik:** Pertimbangkan untuk menganalisis distribusi toksisitas berdasarkan kolom `topic` untuk memahami domain mana yang paling banyak mengandung *hate speech*.

---
## Bab 6: Ekspor Data Interim
Kode di bawah ini menyimpan dataframe yang sudah diparsing labelnya ke `data/interim/data_parsed.csv`.  
> ⚠️ **Kode sengaja di-comment karena saat ini masih fokus pada tahap EDA.** Uncomment dan jalankan ketika siap melanjutkan ke tahap Data Engineering.

In [ ]:
# ============================================================
# KODE EKSPOR DATA INTERIM
# Uncomment (hapus tanda #) ketika siap melanjutkan ke
# tahap Data Engineering / Cleaning.
# ============================================================

# import os
# 
# # Pilih kolom yang relevan untuk disimpan
# export_cols = [
#     'text_id', 'text', 'initial_paragraph', 'topic',
#     'toxicity_label', 'identity_attack_label',
#     'threat_incitement_to_violence_label', 'insults_label',
#     'profanity_obscenity_label', 'sexually_explicit_label',
#     'polarized_label', 'related_to_election_2024_label',
#     'is_noise_or_spam_text_label',
#     'char_count', 'word_count', 'num_annotators'
# ]
# 
# df_export = df[export_cols].copy()
# 
# # Buat folder jika belum ada
# os.makedirs('data/interim', exist_ok=True)
# 
# # Simpan ke CSV
# df_export.to_csv('data/interim/data_parsed.csv', index=False)
# print(f"Data berhasil disimpan ke data/interim/data_parsed.csv")
# print(f"   Jumlah baris: {len(df_export):,}")
# print(f"   Jumlah kolom: {len(df_export.columns)}")

---
## Bab 5: Advanced NLP Analysis
Bagian ini memuat analisis khusus NLP (seperti N-Gram, TF-IDF, dan Lexical Diversity) untuk memahami karakteristik teks lebih dalam sebelum masuk ke tahap pemodelan.

In [ ]:
# Setup variables yang dibutuhkan oleh analisis lanjutan
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
text_col = 'text'
main_label_col = 'toxicity_label'
eda_text = df[text_col].fillna('').copy()
indo_stopwords = set([
    'yang', 'di', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'dari', 'ke', 'pada',
    'dalam', 'adalah', 'juga', 'tidak', 'ada', 'orang', 'yg', 'ya', 'aja',
    'kalo', 'buat', 'sama', 'bisa', 'karena', 'kalau', 'akan', 'aku', 'saya',
    'dia', 'mereka', 'kita', 'kamu', 'udah', 'gak', 'nya', 'kok', 'sih', 'lagi',
    'lebih', 'banyak', 'sudah', 'baru', 'jadi'
])
stopwords_id = indo_stopwords
from collections import Counter
all_tokens = re.findall(r"(?u)\b[\w']+\b", " ".join(eda_text.astype(str).str.lower()))
all_counter = Counter(all_tokens)
print('Setup berhasil!')

### 5.3 Data Duplikat, Conflicting Labels, dan Potensi Leakage
Duplikat tidak langsung dihapus. Analisis membedakan duplikat seluruh baris, duplikat teks, konflik label pada teks identik, dan duplikasi setelah normalisasi ringan sebagai indikator *near-duplicate*.

In [ ]:
import re

n_dup_rows = int(df.duplicated().sum())
n_dup_text = int(df.duplicated(subset=[text_col], keep=False).sum())
n_dup_text_excess = int(df.duplicated(subset=[text_col]).sum())

def normalize_for_dup(x):
    x = str(x).lower()
    x = re.sub(r'https?://\S+|www\.\S+', ' URL ', x)
    x = re.sub(r'@\w+', ' USER ', x)
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'[^\w\s]', '', x)
    return x.strip()

eda_norm_text = df[text_col].fillna('').map(normalize_for_dup)
n_norm_dup_excess = int(eda_norm_text.duplicated().sum())

dup_summary = pd.DataFrame({
    'Metrik': ['Duplicate rows identik', 'Baris yang termasuk grup teks duplikat', 'Duplikat teks berlebih', 'Duplikat setelah normalisasi ringan (indikasi near-duplicate)'],
    'Jumlah': [n_dup_rows, n_dup_text, n_dup_text_excess, n_norm_dup_excess]
})
dup_summary['Persentase'] = (dup_summary['Jumlah']/len(df)*100).round(3)
display(dup_summary)

if n_dup_text > 0:
    print("Contoh teks duplikat:")
    display(df[df.duplicated(subset=[text_col], keep=False)][[c for c in ['text_id', text_col, main_label_col] if c in df.columns]].sort_values(text_col).head(12))

conflict_examples = pd.DataFrame()
conflict_text_count = 0
if main_label_col:
    tmp_conf = df[[text_col, main_label_col]].dropna()
    conflict_keys = tmp_conf.groupby(text_col)[main_label_col].nunique()
    conflict_keys = conflict_keys[conflict_keys > 1].index
    conflict_text_count = len(conflict_keys)
    if conflict_text_count:
        conflict_examples = df[df[text_col].isin(conflict_keys)][[c for c in ['text_id', text_col, main_label_col] if c in df.columns]].sort_values(text_col)
        print(f"Teks identik dengan conflicting labels pada target utama: {conflict_text_count:,} teks unik")
        display(conflict_examples.head(12))
    else:
        print("Tidak ditemukan teks identik dengan label target utama yang berbeda.")

print("\nREKOMENDASI:")
print("Duplikat dapat menaikkan bobot frekuensi secara tidak proporsional, mencerminkan spam, dan menimbulkan data leakage bila salinan/near-duplicate terpisah antara train dan test. Evaluasi dan deduplikasi sebaiknya dilakukan sebelum train-test split, sambil meninjau kasus conflicting labels secara manual atau dengan aturan konsensus yang konsisten.")

### 5.6 Karakter Khusus dan Elemen Digital
Untuk hate speech, elemen emosional tidak otomatis dihapus. Analisis ini mengukur prevalensinya terlebih dahulu.

In [ ]:
patterns = {
    'URL': r'https?://\S+|www\.\S+',
    'Mention @username': r'@\w+',
    'Hashtag': r'#\w+',
    'Angka': r'\d',
    'Emoji (heuristik Unicode)': u'[\U0001F300-\U0001FAFF\u2600-\u27BF]',
    'Tanda baca berlebihan': r'[!]{3,}|[?]{3,}|[.]{3,}|[,]{3,}',
    'Huruf kapital berlebihan': r'\b[A-Z]{4,}\b',
    'Newline': r'\n|\r',
    'Karakter non-alfanumerik': r'[^\w\s]'
}
rows=[]
for name, pat in patterns.items():
    mask=eda_text.str.contains(pat, regex=True, na=False)
    rows.append({'Elemen':name,'Jumlah Dokumen':int(mask.sum()),'Persentase':round(mask.mean()*100,3)})
element_table=pd.DataFrame(rows)
display(element_table)

recommend_map = {
    'URL':'ganti token khusus <URL> bila cukup sering; isi URL biasanya tidak perlu dipertahankan',
    'Mention @username':'ganti token <USER> untuk mengurangi sparsity tanpa kehilangan sinyal adanya mention',
    'Hashtag':'pertahankan isi hashtag atau pisahkan tanda #; dapat membawa topik/stance penting',
    'Angka':'pertahankan atau normalisasi <NUM> sesuai konteks; jangan hapus otomatis',
    'Emoji (heuristik Unicode)':'pertahankan atau konversi ke deskripsi/token emoji karena bisa membawa emosi/sarkasme',
    'Tanda baca berlebihan':'normalisasi jumlahnya tetapi pertahankan sinyal intensitas',
    'Huruf kapital berlebihan':'jika lowercase dilakukan, pertimbangkan fitur/token penanda ALLCAPS agar intensitas tidak hilang',
    'Newline':'normalisasi ke spasi bila tidak bermakna struktural',
    'Karakter non-alfanumerik':'bersihkan selektif; jangan menghapus emoji/hashtag/punctuation emosional secara membabi buta'
}
rec_df=element_table.copy(); rec_df['Rekomendasi']=rec_df['Elemen'].map(recommend_map)
display(rec_df)


### 5.7 Bahasa, Slang, Singkatan, Variasi Penulisan, dan Typo
Bagian ini bersifat eksploratif. Kandidat ditampilkan hanya bila benar-benar ditemukan di corpus; tidak ada normalisasi permanen.

In [ ]:
from collections import Counter

basic_tokens = re.findall(r"(?u)\b[\w']+\b", ' '.join(eda_text.str.lower()))
basic_counter = Counter(basic_tokens)

slang_map = {
    'gak':'tidak','ga':'tidak','nggak':'tidak','tdk':'tidak','tak':'tidak',
    'yg':'yang','dgn':'dengan','dr':'dari','krn':'karena','karna':'karena','kalo':'kalau','kl':'kalau',
    'udh':'sudah','udah':'sudah','blm':'belum','bgt':'banget','banget':'sangat','aja':'saja','sm':'sama',
    'gw':'saya/aku','gue':'saya/aku','lu':'kamu','lo':'kamu','org':'orang','jd':'jadi','jgn':'jangan'
}
slang_rows=[{'Bentuk Asli':w,'Frekuensi':basic_counter[w],'Dugaan Normalisasi':norm} for w,norm in slang_map.items() if basic_counter[w]>0]
slang_df=pd.DataFrame(sorted(slang_rows,key=lambda x:x['Frekuensi'],reverse=True))
print("Kandidat slang/singkatan yang benar-benar ditemukan:")
display(slang_df.head(30) if len(slang_df) else pd.DataFrame(columns=['Bentuk Asli','Frekuensi','Dugaan Normalisasi']))

elong_counter=Counter()
for tok,c in basic_counter.items():
    if re.search(r'(.)\1{2,}', tok): elong_counter[tok]=c
elong_df=pd.DataFrame(elong_counter.most_common(20), columns=['Bentuk Memanjang/Variasi','Frekuensi'])
print("Kandidat kata memanjang/typo berbasis repeated-character:")
display(elong_df)

english_markers={'the','and','is','are','you','your','this','that','of','to','for','with','not','fuck','shit','stupid'}
found_en=[(w,basic_counter[w]) for w in english_markers if basic_counter[w]>0]
print("Marker bahasa Inggris yang ditemukan (indikasi code-mixing, bukan deteksi bahasa formal):")
display(pd.DataFrame(sorted(found_en,key=lambda x:x[1],reverse=True), columns=['Token','Frekuensi']))

if len(slang_df):
    ex_words=set(slang_df['Bentuk Asli'].head(10))
    mask=eda_text.str.lower().apply(lambda s:any(re.search(rf'\b{re.escape(w)}\b',s) for w in ex_words))
    print("Contoh nyata teks yang mengandung kandidat slang/singkatan:")
    display(df.loc[mask,[c for c in ['text_id',text_col,main_label_col] if c in df.columns]].head(8))
print("Catatan: kandidat typo tidak dapat dipastikan hanya dari statistik corpus. Normalisasi perlu kamus/aturan yang divalidasi agar kata ofensif, nama, dan slang penting tidak berubah makna.")

### 5.10 Vocabulary Size dan Lexical Diversity

In [ ]:
total_tokens=len(all_tokens); unique_tokens=len(all_counter)
once=sum(1 for c in all_counter.values() if c==1); twice=sum(1 for c in all_counter.values() if c==2)
lex_div=unique_tokens/total_tokens if total_tokens else np.nan
vocab_stats=pd.DataFrame({
    'Metrik':['Total token','Unique token / vocabulary size','Lexical diversity','Token muncul 1 kali','Token muncul 2 kali'],
    'Nilai':[total_tokens,unique_tokens,round(lex_div,4),once,twice]
})
display(vocab_stats)
print(f"Vocabulary per dokumen = {unique_tokens/max(len(df),1):.3f} unique token/dokumen (indikator kasar).")
if total_tokens:
    print(f"Hapax legomena (muncul sekali) mencakup {once/unique_tokens*100:.2f}% dari vocabulary." if unique_tokens else '')

### 5.11 Rare Words Analysis
Rare words tidak otomatis dihapus karena bisa memuat kata ofensif spesifik, nama target, slang, atau bentuk khas komunitas.

In [ ]:
rare1=[w for w,c in all_counter.items() if c==1]
rare2=[w for w,c in all_counter.items() if c==2]
rare5=[w for w,c in all_counter.items() if c<=5]
rare_stats=pd.DataFrame({'Frekuensi':['= 1','= 2','≤ 5'],'Jumlah token unik':[len(rare1),len(rare2),len(rare5)]})
display(rare_stats)
print("Contoh token frekuensi 1:", rare1[:30])
print("Contoh token frekuensi 2:", rare2[:30])

rare_examples=[]
for w in rare5[:500]:
    category=[]
    if re.search(r'\d',w): category.append('kode/angka')
    if re.search(r'(.)\1{2,}',w): category.append('typo/elongasi')
    if w in slang_map: category.append('slang/singkatan')
    if len(w)>=15: category.append('token sangat panjang/nama/URL residual')
    if category: rare_examples.append((w,all_counter[w],', '.join(category)))
rare_class_df=pd.DataFrame(rare_examples[:30],columns=['Rare Word','Frekuensi','Kemungkinan Asal (heuristik)'])
display(rare_class_df)
print("Interpretasi: heuristik hanya membantu penyaringan kandidat. Rare words harus ditinjau sebelum dibuang karena kata target kelompok atau makian tertentu justru dapat jarang tetapi sangat informatif.")

### 5.12 N-Gram Analysis: Unigram, Bigram, Trigram
N-gram dihitung dari teks dengan pembersihan ringan dan stopword konservatif. Ini membantu melihat frasa yang mungkin lebih informatif daripada token tunggal.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def lightly_clean(s):
    s=str(s).lower(); s=re.sub(r'https?://\S+|www\.\S+',' URL ',s); s=re.sub(r'@\w+',' USER ',s); s=re.sub(r'\s+',' ',s); return s.strip()
eda_clean_text=eda_text.map(lightly_clean)

def top_ngrams(texts, ngram_range, top_n=20, min_df=2):
    vec=CountVectorizer(ngram_range=ngram_range, stop_words=list(stopwords_id), token_pattern=r'(?u)\b\w+\b', min_df=min_df, max_features=50000)
    X=vec.fit_transform(texts)
    sums=np.asarray(X.sum(axis=0)).ravel(); terms=np.array(vec.get_feature_names_out())
    idx=np.argsort(sums)[-top_n:][::-1]
    return pd.DataFrame({'N-Gram':terms[idx],'Frekuensi':sums[idx].astype(int)})

uni_top=top_ngrams(eda_clean_text,(1,1)); bi_top=top_ngrams(eda_clean_text,(2,2)); tri_top=top_ngrams(eda_clean_text,(3,3))
for title,tab in [('Top 20 Unigram',uni_top),('Top 20 Bigram',bi_top),('Top 20 Trigram',tri_top)]:
    print(title); display(tab)
    fig,ax=plt.subplots(figsize=(10,6)); ax.barh(tab['N-Gram'][::-1],tab['Frekuensi'][::-1]); ax.set_title(title); ax.set_xlabel('Frekuensi'); ax.set_ylabel('N-Gram'); plt.tight_layout(); plt.show()

if main_label_col:
    per_class_bigrams={}
    for lab in sorted(df[main_label_col].dropna().unique()):
        texts=eda_clean_text[df[main_label_col].eq(lab)]
        if len(texts)>=2:
            try: per_class_bigrams[str(lab)]=top_ngrams(texts,(2,2),10,min_df=1)
            except ValueError: pass
    for lab,tab in per_class_bigrams.items():
        print(f"Top bigram label {lab}:"); display(tab)
print("Interpretasi: bigram/trigram yang konsisten per kelas dapat menangkap konteks yang hilang pada unigram, termasuk negasi atau target ujaran.")

### 5.13 TF-IDF Analysis
TF-IDF menekankan term yang relatif khas pada dokumen/corpus, berbeda dari frekuensi mentah yang hanya menghitung kemunculan.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec=TfidfVectorizer(stop_words=list(stopwords_id), token_pattern=r'(?u)\b\w+\b', min_df=2, max_df=0.98, max_features=30000, sublinear_tf=True)
X_tfidf=tfidf_vec.fit_transform(eda_clean_text)
terms=np.array(tfidf_vec.get_feature_names_out())
mean_weights=np.asarray(X_tfidf.mean(axis=0)).ravel()
idx=np.argsort(mean_weights)[-20:][::-1]
top_tfidf=pd.DataFrame({'Term':terms[idx],'Mean TF-IDF':mean_weights[idx]})
print(f"Ukuran matriks TF-IDF: {X_tfidf.shape}")
print(f"Jumlah fitur: {len(terms):,}")
display(top_tfidf)
fig,ax=plt.subplots(figsize=(10,6)); ax.barh(top_tfidf['Term'][::-1],top_tfidf['Mean TF-IDF'][::-1]); ax.set_title('Top 20 Term berdasarkan Mean TF-IDF'); ax.set_xlabel('Mean TF-IDF'); ax.set_ylabel('Term'); plt.tight_layout(); plt.show()

per_class_tfidf={}
if main_label_col:
    labels_sorted=sorted(df[main_label_col].dropna().unique())
    for lab in labels_sorted:
        mask=df[main_label_col].eq(lab).to_numpy()
        if mask.sum():
            cw=np.asarray(X_tfidf[mask].mean(axis=0)).ravel(); ci=np.argsort(cw)[-15:][::-1]
            per_class_tfidf[str(lab)]=pd.DataFrame({'Term':terms[ci],'Mean TF-IDF':cw[ci]})
            print(f"Top TF-IDF label {lab}:"); display(per_class_tfidf[str(lab)])
print("Perbedaan: word frequency menunjukkan kata yang paling sering; TF-IDF menurunkan pengaruh term yang terlalu umum dan menaikkan term yang lebih khas. Pada tahap ini keduanya hanya digunakan sebagai EDA.")